# Vision-Language Models (VLMs)

> Send a model a photo of a refrigerator and ask what to cook. How do pixels become a sequence a language model can process?
>
> A 224×224 image divided into 16×16 patches gives 14×14=196 patches. Flattening and projecting each patch turns a pixel grid into a sequence of visual vectors.
>
> This chapter covers four topics:
>
> 1. **Images to tokens**: patching, a vision encoder, and a projector supply visual representations to the language model.
> 2. **Fusion architectures**: token concatenation, cross-attention, and Q-Former provide different connections between image and text.
> 3. **Training and alignment**: staged freezing, visual instruction tuning, and CLIP-style dual-encoder contrastive learning.
> 4. **Engineering cost**: resolution affects visual sequence length, attention work, and caches.


## 1. What a Language Model Can Receive

A text tokenizer converts words, characters, or fragments into token IDs. An **embedding** maps an ID such as 12 into a vector like `[0.3, -0.1, ...]`. The model computes with vectors, not the integer's numerical magnitude.

An image instead begins as a two-dimensional pixel grid:

```text
Text: "a cat on a mat"
  → tokenizer → [12, 45, 78, 3, 90, 23]
  → embedding → six vectors
Image: 224×224 pixels
  → ? → vectors the language model can use
```

The first problem is therefore how to represent an image as embeddings. A simple approach divides it into local patches, treating each as a visual unit in a sequence. These continuous visual tokens need not have discrete IDs in the text vocabulary.


## 2. Image Patches

**Patchify** means dividing an image into patches. With a 224×224 image and 16×16 patches, each side has 224/16=14 patches, for 196 total.

```text
Image: 224×224
Patch: 16×16
Grid:  14×14 = 196 patches

┌──┬──┬──┬──┐
│  │  │  │  │
├──┼──┼──┼──┤
│  │cat│ │  │ ← a face can span several patches
├──┼──┼──┼──┤
│  │  │  │  │
└──┴──┴──┴──┘
```

This is a classic [ViT](https://arxiv.org/abs/2010.11929) configuration, rather than a universal VLM choice. Models can use patch sizes such as 14, 16, or 32, image resolutions such as 336, multiple crops, or dynamic resolution.

A single vector for the whole image may lose local detail. Patches retain information about smaller regions. The following `unfold` example inspects their shape without a pretrained vision model.


In [ ]:
# First calculate the shape by hand, then verify patchify in code
import torch
import torch.nn as nn

torch.manual_seed(42)

IMG_SIZE = 224
PATCH_SIZE = 16

# Simulate one RGB image: [channels, height, width]
fake_image = torch.randn(3, IMG_SIZE, IMG_SIZE)

print("Original image shape:", fake_image.shape)
print("Explanation: three color channels, each a 224 by 224 grid")

# unfold slices windows along height and width; stride=16 makes them non-overlapping
patches = fake_image.unfold(1, PATCH_SIZE, PATCH_SIZE)
patches = patches.unfold(2, PATCH_SIZE, PATCH_SIZE)

print("\nShape after patching:", patches.shape)
print("Explanation: [3 channels, 14 patch rows, 14 patch columns, 16 height, 16 width]")

num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
patch_dim = 3 * PATCH_SIZE * PATCH_SIZE
patches_flat = patches.permute(1, 2, 0, 3, 4).reshape(num_patches, patch_dim)

print("\nShape after flattening:", patches_flat.shape)
print(f"Key observation: one image becomes {num_patches} patches")
print(f"Each patch contains {patch_dim} numbers, or 3 x 16 x 16")


## 3. Patch Embedding

An RGB 16×16 patch contains $3×16×16=768$ pixel values. A shared projection maps them into a vector:

```text
Patch:             768 values
Patch embedding:   Linear(768 → d_model)
Output:            one d_model-dimensional vector
Text embedding:    one d_model-dimensional vector
```

The two representations need the same width for concatenation. In a real VLM, the vision encoder may first use its own width, followed by a projector to the LLM width.

A `Conv2d` with `kernel_size=patch_size` and `stride=patch_size` efficiently performs patch extraction and the shared linear projection in one operation. Here the convolution implements patch embedding, rather than a complete multi-layer CNN feature extractor.


In [ ]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    """Split an image into patches and map every patch to a visual Token."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=512):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        # kernel_size=patch_size: inspect one patch at a time
        # stride=patch_size: move the next window exactly to the next patch
        self.proj = nn.Conv2d(
            in_channels,
            d_model,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        """
        Parameters:
            x: images of shape [batch, 3, 224, 224]
        Returns:
            visual Tokens of shape [batch, num_patches, d_model]
        """
        x = self.proj(x)      # [batch, d_model, 14, 14]
        x = x.flatten(2)      # [batch, d_model, 196]
        x = x.transpose(1, 2) # [batch, 196, d_model]
        return x

patch_emb = PatchEmbedding(img_size=224, patch_size=16, d_model=512)
dummy_img = torch.randn(2, 3, 224, 224)
visual_tokens = patch_emb(dummy_img)

print("Input image:", dummy_img.shape)
print("Visual Tokens:", visual_tokens.shape)
print("Key observation: each image in the batch produces 196 visual Tokens, each with 512 dimensions.")


## 4. Combining Visual and Text Tokens

Our patch projection produces `[batch, 196, d_model]`, while text embeddings have shape `[batch, text_len, d_model]`. Concatenate along sequence length:

```text
[vis_1] ... [vis_196] [please] [describe] [this] [image]
```

This gives the intuition behind a LLaVA-style visual-token approach: the LLM receives a sequence of embeddings. Actual templates use an image placeholder, boundary markers, or insertion positions to replace the appropriate portion with visual features. The number of inserted tokens depends on the vision encoder and connector, not always the raw patch count.


In [ ]:
# Simulate concatenating visual and text Tokens
import torch
import torch.nn as nn

d_model = 512
text_vocab_size = 1000

# Assume the tokenizer maps “describe this image” to five IDs
text_ids = torch.tensor([[5, 12, 78, 3, 90]])
text_embedding = nn.Embedding(text_vocab_size, d_model)
text_tokens = text_embedding(text_ids)

visual_tokens = torch.randn(1, 196, d_model)
combined = torch.cat([visual_tokens, text_tokens], dim=1)

print("Visual Tokens:", visual_tokens.shape)
print('"Text token:"', text_tokens.shape)
print("After concatenation:", combined.shape)
print("Key observation: 196 image tokens plus five text tokens equals 201 tokens.")


## 5. The Visual Projector

Our teaching `PatchEmbedding` maps pixels directly to `d_model`. Real VLMs commonly use a pretrained vision encoder first, then a **projector**:

```text
Image → Vision encoder → [B, 196, d_vis]
                       → Projector → [B, 196, d_llm]
Text → Token embedding             → [B, T, d_llm]
Concatenated sequence: [B, 196+T, d_llm]
```

The encoder and LLM do not naturally share an embedding space. Training the connector makes visual features useful to the language model; merely matching dimensions does not establish semantic alignment.

Early [LLaVA](https://arxiv.org/abs/2304.08485) uses a linear connector; [LLaVA-1.5](https://openaccess.thecvf.com/content/CVPR2024/papers/Liu_Improved_Baselines_with_Visual_Instruction_Tuning_CVPR_2024_paper.pdf) commonly uses a two-layer MLP. Other designs use Q-Former, Perceiver resampling, cross-attention, or other connectors.


In [ ]:
import torch
import torch.nn as nn

class LLaVAProjector(nn.Module):
    """Map visual features into the LLM embedding space."""
    def __init__(self, vis_dim=1024, llm_dim=4096):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(vis_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, visual_features):
        """
        Parameters:
            visual_features: [batch, num_patches, vis_dim]
        Returns:
            [batch, num_patches, llm_dim]
        """
        return self.mlp(visual_features)

projector = LLaVAProjector(vis_dim=1024, llm_dim=4096)
vision_features = torch.randn(2, 196, 1024)
llm_space_features = projector(vision_features)

print("Vision Encoder output:", vision_features.shape)
print("Projector output:", llm_space_features.shape)
print("Key observation: the patch count is unchanged; only each Token's dimension is aligned with the LLM.")


## 6. Three Common Architecture Patterns

These are useful teaching categories; real systems can combine them:

| Pattern | Image connection | Changes to the LLM | Example |
|:---|:---|:---|:---|
| Visual tokens | Insert visual embeddings into the sequence | Core language stack can remain intact | LLaVA |
| Cross-attention | Text states query visual features | Add attention blocks | Flamingo |
| Q-Former | Learned queries extract compact visual representations | Can connect to a frozen LLM | BLIP-2 |

### 6.1 Visual Tokens

In our example, 196 visual tokens precede text. This reuses the language stack but increases sequence length and attention/cache cost.

### 6.2 Cross-Attention

Self-attention obtains Q/K/V from one sequence. Cross-attention uses text queries and visual keys/values:

$$\operatorname{CrossAttn}(X_{text},X_{vis})=
\operatorname{softmax}\left(\frac{Q_{text}K_{vis}^T}{\sqrt{d_k}}\right)V_{vis}.$$

[Flamingo](https://arxiv.org/abs/2204.14198) first resamples images into a fixed number of visual latents, then inserts gated cross-attention into the language model. Gates help control the added branch. Our toy block also applies a LayerNorm after the residual, so a zero gate does not make its complete output exactly equal to its input.

### 6.3 Q-Former

[BLIP-2](https://arxiv.org/abs/2301.12597) uses learned queries—32 in its reported setup—to extract a compact representation from image features. This can reduce the visual tokens passed to the LLM. The query count is a design choice, not a fixed theoretical value.

The next example compares rough attention-score counts for token concatenation and text self-attention plus visual cross-attention. It is not a complete FLOP or latency benchmark; vision encoding, projections, heads/layers, masking, and kernels also matter.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossAttention(nn.Module):
    """Cross-Attention: Q comes from text, while K and V come from the image."""
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, text_hidden, image_features):
        """
        Parameters:
            text_hidden: [batch, text_len, d_model]
            image_features: [batch, num_visual, d_model]
        Returns:
            [batch, text_len, d_model]
        """
        batch, text_len, d_model = text_hidden.shape
        num_visual = image_features.shape[1]

        query = self.q_proj(text_hidden)
        key = self.k_proj(image_features)
        value = self.v_proj(image_features)

        query = query.view(batch, text_len, self.n_heads, self.head_dim)
        query = query.transpose(1, 2)
        key = key.view(batch, num_visual, self.n_heads, self.head_dim)
        key = key.transpose(1, 2)
        value = value.view(batch, num_visual, self.n_heads, self.head_dim)
        value = value.transpose(1, 2)

        scores = (query @ key.transpose(-2, -1)) * self.scale
        weights = F.softmax(scores, dim=-1)
        out = weights @ value
        out = out.transpose(1, 2).reshape(batch, text_len, d_model)
        return self.out_proj(out)

class GatedCrossAttentionBlock(nn.Module):
    """A small Cross-Attention module with a tanh gate."""
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        self.cross_attn = CrossAttention(d_model, n_heads)
        self.norm = nn.LayerNorm(d_model)
        self.alpha = nn.Parameter(torch.zeros(1))

    def forward(self, text_hidden, image_features):
        gate = torch.tanh(self.alpha)
        x = text_hidden + gate * self.cross_attn(text_hidden, image_features)
        return self.norm(x)

batch = 1
text_len = 50
num_visual = 196
d_model = 512

text_hidden = torch.randn(batch, text_len, d_model)
image_features = torch.randn(batch, num_visual, d_model)

visual_token_len = text_len + num_visual
visual_token_cost = visual_token_len ** 2
cross_attn_cost = text_len ** 2 + text_len * num_visual

block = GatedCrossAttentionBlock(d_model=d_model, n_heads=8)
out = block(text_hidden, image_features)

print("Visual Token sequence length:", visual_token_len)
print("Approximate Visual Token Attention cost:", visual_token_cost)
print("Cross-Attention output shape:", out.shape)
print("Approximate Cross-Attention cost:", cross_attn_cost)
print("Initial gate tanh(alpha):", torch.tanh(block.alpha).item())
print("Key observation: Cross-Attention does not lengthen the text sequence, but it requires modifying the LLM architecture.")


## 7. Image Compute Costs

A 224×224 image with 16×16 patches contributes 196 positions in the simple example. A short image-description prompt may contain far fewer text tokens; long conversations, multiple images, and video change this balance.

More visual tokens increase full-attention prefill work, approximately quadratic in combined length, as well as cache use and per-token decoding work. It is not correct to describe every cached decoding step as quadratic in the complete sequence length.

Compressing visual tokens can discard detail, especially small text, crowded objects, or charts. Resampling, dynamic crops, and multiple scales aim to retain task-relevant information while managing cost.


In [ ]:
# Intuition: larger patches produce fewer visual Tokens
for patch_size in [8, 14, 16, 32]:
    tokens_per_side = 224 // patch_size
    num_tokens = tokens_per_side ** 2
    print(f"patch_size={patch_size:2d}: {tokens_per_side:2d} x {tokens_per_side:2d}",
          f"= {num_tokens:4d} visual Tokens")

print("\nKey observation: patch size 32 gives only 49 tokens but coarser detail.")
print("Key observation: patch size 8 gives 784 tokens and more detail, but slower inference.")


## 8. Training with Frozen Components

LLaVA/BLIP-2-style reuse starts from pretrained vision and language components, rather than training every large module from scratch:

```text
Vision encoder: a pretrained model such as CLIP ViT, SigLIP, or EVA-CLIP.
                ViT is an architecture, not itself a particular checkpoint.
Projector:      a connector to learn between visual and language features.
LLM:            a pretrained language model.
```

Freezing saves optimizer/gradient storage and can help preserve useful representations when data is limited. A classic LLaVA-style recipe is:

| Stage | Train | Freeze | Goal |
|:---|:---|:---|:---|
| Feature alignment | Projector | Vision encoder + LLM | Make visual features useful to the LLM |
| Instruction tuning | Projector + LLM | Vision encoder | Learn visual question answering and assistant behavior |

Other recipes unfreeze the vision tower or use additional stages. Setting `requires_grad=False` freezes parameter updates, while the parameters still participate in the forward pass. A frozen LLM may still need an autograd graph to propagate gradients back into a trainable projector.


In [ ]:
# Simulate VLM freezing strategies with three small modules
import torch.nn as nn

vision_encoder = nn.Linear(1024, 1024)
projector = LLaVAProjector(vis_dim=1024, llm_dim=4096)
llm_head = nn.Linear(4096, 1000)

def set_trainable(module, trainable):
    """Set a module to trainable or frozen."""
    for param in module.parameters():
        param.requires_grad = trainable

def count_trainable(module):
    """Count trainable parameters in a module."""
    return sum(param.numel() for param in module.parameters() if param.requires_grad)

def show_stage(stage):
    """Display which components are frozen in a particular training stage."""
    if stage == 1:
        set_trainable(vision_encoder, False)
        set_trainable(projector, True)
        set_trainable(llm_head, False)
    else:
        set_trainable(vision_encoder, False)
        set_trainable(projector, True)
        set_trainable(llm_head, True)

    print(f"Stage {stage}")
    print("  Trainable Vision Encoder parameters:", count_trainable(vision_encoder))
    print("  Trainable Projector parameters:     ", count_trainable(projector))
    print("  Trainable LLM Head parameters:      ", count_trainable(llm_head))

show_stage(1)
print()
show_stage(2)
print("\nKey observation: the Vision Encoder stays frozen in both stages, while the Projector trains in both.")


## 9. Engineering Details

**Where does the image begin?** A format may delimit its visual embeddings:

```text
<image_start> [vis_1] ... [vis_196] <image_end> describe this image
```

Boundary markers and placeholders help organize multiple images and turns; exact conventions are checkpoint-specific.

**What positional information do visual tokens use?** A simple row-major sequence numbers patches as:

```text
Row 0: patch 0, 1, ..., 13
Row 1: patch 14, 15, ..., 27
```

Vision encoders normally supply spatial information, and the language-side sequence has its own positional scheme. Real models may use two-dimensional positions, multiple scales, or dynamic resolution.

**How are high-resolution images handled?** Resizing a 4K image to 224×224 loses small text and detail. LLaVA-NeXT/LLaVA-1.6's AnyRes approach combines a global view with encoded local crops, at the cost of more visual tokens.


## 10. A Minimal VLM

1. `PatchEmbedding` produces visual vectors.
2. `text_embedding` maps text IDs to vectors.
3. `torch.cat` joins the two sequences.
4. `TransformerEncoder` processes them.
5. `lm_head` produces a vocabulary distribution at each position.

This MiniVLM demonstrates tensor flow only. It is untrained, has no real tokenizer, uses bidirectional attention without a causal mask, and omits positional encoding. It is therefore not a working GPT-style conversational generator. A real model needs the appropriate architecture, positions, training, and generation procedure.


In [ ]:
import torch
import torch.nn as nn

class MiniVLM(nn.Module):
    """A minimal Visual Token VLM."""
    def __init__(self, text_vocab_size=100, d_model=128, num_heads=2, num_layers=2):
        super().__init__()
        self.vision = PatchEmbedding(
            img_size=224,
            patch_size=16,
            in_channels=3,
            d_model=d_model,
        )
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            batch_first=True,
        )
        self.layers = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, text_vocab_size)

    def forward(self, images, text_ids):
        """
        Parameters:
            images: [batch, 3, 224, 224]
            text_ids: [batch, text_len]
        Returns:
            [batch, 196 + text_len, vocab_size]
        """
        visual_tokens = self.vision(images)
        text_tokens = self.text_embedding(text_ids)
        tokens = torch.cat([visual_tokens, text_tokens], dim=1)
        hidden = self.layers(tokens)
        hidden = self.norm(hidden)
        return self.lm_head(hidden)

vlm = MiniVLM(text_vocab_size=100, d_model=128, num_heads=2, num_layers=2)

dummy_images = torch.randn(1, 3, 224, 224)
dummy_text = torch.randint(0, 100, (1, 10))
logits = vlm(dummy_images, dummy_text)

print("Input image:", dummy_images.shape)
print("Input text:", dummy_text.shape)
print("Output logits:", logits.shape)
print("Key observation: 196 visual positions plus 10 text positions gives 206 output positions.")
print("Total parameters:", f"{sum(param.numel() for param in vlm.parameters()):,}")


## 11. CLIP-Style Image–Text Alignment

A connector must learn how visual features relate to language. [CLIP](https://arxiv.org/abs/2103.00020) demonstrates a related alignment mechanism: contrastive training brings matched image/text representations together in a shared space and separates mismatches.

CLIP-style pretrained visual features are useful to systems such as early LLaVA and the original Qwen-VL. They are not the only VLM front end: other systems use SigLIP or separately trained visual encoders. CLIP's shared text space is also distinct from an LLM's token-embedding space, so a language connector still needs training.

We train a tiny dual encoder on colored shapes to observe the alignment objective.


### 11.1 Contrastive Learning

For a batch of paired images and descriptions, compute normalized image vectors $I_i$ and text vectors $T_j$. Their cosine similarities form a B×B matrix, with correct pairs on the diagonal. Row-wise cross-entropy encourages each image to select its matching text:

$$L_{i2t}=-\frac1B\sum_i\log\frac{\exp(I_i\cdot T_i/\tau)}{\sum_j\exp(I_i\cdot T_j/\tau)}.$$

CLIP also applies the reverse text-to-image direction and averages the two losses. The next 3×3 example shows one direction; the training loop below uses the symmetric objective.

A smaller temperature makes softmax sharper. Its effect on an individual loss depends on the similarity ordering; it does not always increase every mismatch penalty.


In [ ]:
import numpy as np

# Assume three images and three sentences are encoded, with pairwise cosine similarities ready
# sim[i,j] is image i versus sentence j; diagonal entries are correct pairs
sim = np.array([
    [0.90, 0.20, 0.10],
    [0.10, 0.80, 0.30],
    [0.20, 0.10, 0.85],
])


def softmax_rows(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


temp = 0.1
logits = sim / temp
probs = softmax_rows(logits)          # probability of each sentence for every image

print("Similarity / temperature:")
print(np.round(logits, 1))
print("\nProbability of three sentences for each image; rows should concentrate on the diagonal:")
print(np.round(probs, 3))

ce = -np.log([probs[i, i] for i in range(3)])
print(f"\nPer-pair cross entropy: {np.round(ce, 3)}   mean InfoNCE = {ce.mean():.3f}")
print("\nKey observation: training adjusts the encoders to raise diagonal matched-pair similarity and lower the rest.")
print("  Lower temperature sharpens the distribution and penalizes mismatched pairs more strongly.")


### 11.2 Synthetic Image–Text Pairs

The original CLIP trains on 400 million image/text pairs. Here we generate three colors × three shapes = nine classes with NumPy. Random position and size variation produce multiple images for the same description.

This creates a compact teaching dataset with **known pairings**. Variation discourages memorizing one exact image, but does not by itself prove learned semantic generalization beyond these shapes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

COLORS = ["red", "green", "blue"]
SHAPES = ["circle", "square", "triangle"]
COLOR_RGB = {"red": [1, 0, 0], "green": [0, 1, 0], "blue": [0, 0, 1]}


def make_image(color, shape, size=32, rng=np.random):
    """Draw a colored shape of size×size at a random position."""
    img = np.zeros((size, size, 3), dtype=np.float32)
    cx = rng.randint(10, size - 10)
    cy = rng.randint(10, size - 10)
    r = rng.randint(5, 8)
    yy, xx = np.mgrid[0:size, 0:size]
    if shape == "circle":
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r * r
    elif shape == "square":
        mask = (np.abs(xx - cx) <= r) & (np.abs(yy - cy) <= r)
    else:  # triangle: apex above, base below
        frac = (yy - (cy - r)) / (2 * r)          # 0 at apex, 1 at base
        mask = (yy >= cy - r) & (yy <= cy + r) & (np.abs(xx - cx) <= r * frac)
    img[mask] = COLOR_RGB[color]
    return img


# Text vocabulary: color plus shape plus <pad>
TOK = {w: i for i, w in enumerate(COLORS + SHAPES)}
PAD = len(TOK)
VOCAB = len(TOK) + 1


def make_text(color, shape):
    """Token sequence [color, shape], padded to length four."""
    ids = [TOK[color], TOK[shape]]
    return ids + [PAD] * (4 - len(ids))


# Inspect all nine classes
rng = np.random.RandomState(0)
fig, axes = plt.subplots(3, 3, figsize=(5, 5))
for i, c in enumerate(COLORS):
    for j, s in enumerate(SHAPES):
        axes[i, j].imshow(make_image(c, s, rng=rng))
        axes[i, j].set_title(f"{c} {s}", fontsize=9)
        axes[i, j].axis("off")
plt.suptitle("Synthetic image-text pairs (9 classes)", fontsize=11)
plt.tight_layout()
plt.show()
print(f"Vocabulary: {list(TOK.keys())} + <pad>, {VOCAB} Tokens total")
print("Text encoding example: 'red circle' ->", make_text("red", "circle"))


### 11.3 Dual Encoders

The image tower is a small CNN; the text tower uses embeddings and mean pooling. Both output d-dimensional vectors. Original CLIP studied ResNet and ViT image encoders and uses a Transformer text encoder, at a much larger scale.

The two towers encode inputs independently and interact through the contrastive objective. No fixed-class classifier labels are needed, but training does require knowing which image and text are paired. Our text mean includes a fixed number of padding positions; this is a toy simplification, not CLIP's text pooling.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
torch.set_num_threads(4)


class ImageEncoder(nn.Module):
    """Small CNN mapping an image to a d-dimensional vector."""
    def __init__(self, d=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32→16
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16→8
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 8→4
        )
        self.out = nn.Linear(32 * 4 * 4, d)

    def forward(self, x):                 # x: [B, 3, 32, 32]
        return self.out(self.net(x).flatten(1))


class TextEncoder(nn.Module):
    """Embedding plus mean pooling to a d-dimensional vector."""
    def __init__(self, d=32):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, 16)
        self.out = nn.Linear(16, d)

    def forward(self, ids):               # ids: [B, 4]
        return self.out(self.emb(ids).mean(dim=1))


img_enc = ImageEncoder()
txt_enc = TextEncoder()
n_params = sum(p.numel() for p in img_enc.parameters()) \
         + sum(p.numel() for p in txt_enc.parameters())
print(f"Total dual-encoder parameters: {n_params:,}")


### 11.4 InfoNCE Training

Each step samples a batch of pairs, computes all similarities, and increases the diagonal match probabilities. Our symmetric objective uses fixed temperature $\tau=0.07$; original CLIP learns its logit scale.

The loop keeps the original 300 steps, batch size 18, and learning rate 0.003. Repeated class descriptions create false negatives: two images with the same caption are treated as competing pairs. A larger batch does not eliminate that ambiguity; a multi-positive objective is one possible alternative.


In [ ]:
import torch.nn.functional as F


def info_nce(img_emb, txt_emb, temp=0.07):
    """Symmetric InfoNCE: average image-to-text and text-to-image losses."""
    img_emb = F.normalize(img_emb, dim=-1)          # unit vectors, so dot product is cosine similarity
    txt_emb = F.normalize(txt_emb, dim=-1)
    logits = img_emb @ txt_emb.t() / temp           # [B, B]
    targets = torch.arange(logits.size(0))
    loss_i2t = F.cross_entropy(logits, targets)     # each image finds its description
    loss_t2i = F.cross_entropy(logits.t(), targets)  # each description finds its image
    return (loss_i2t + loss_t2i) / 2


def make_batch(B, rng):
    pairs = [(rng.choice(COLORS), rng.choice(SHAPES)) for _ in range(B)]
    imgs = np.stack([make_image(c, s, rng=rng) for c, s in pairs])
    imgs = torch.tensor(imgs).permute(0, 3, 1, 2)            # [B,3,32,32]
    ids = torch.tensor([make_text(c, s) for c, s in pairs])  # [B,4]
    return imgs, ids


opt = torch.optim.AdamW(
    list(img_enc.parameters()) + list(txt_enc.parameters()), lr=3e-3)

rng = np.random.RandomState(0)
print('"=== CLIP training ==="')
history = []
for step in range(300):
    imgs, ids = make_batch(18, rng)
    loss = info_nce(img_enc(imgs), txt_enc(ids))
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(loss.item())
    if step % 60 == 0 or step == 299:
        print(f"  step {step:3d}  loss {loss.item():.4f}")

print("\nKey observation: loss falls from about 3.2 to about 1; it does not reach zero because some batch items share a class.")
print("Repeated captions create false negatives; larger batches do not remove them.")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3.5))
plt.plot(history, color="steelblue", linewidth=1)
plt.xlabel("training step")
plt.ylabel("InfoNCE loss")
plt.title("CLIP contrastive loss during training")
plt.grid(True, alpha=0.3)
plt.show()

### 11.5 Image-to-Text Retrieval

Give the trained encoder a newly generated image and retrieve the most similar description among the nine classes. The code reports accuracy over 100 queries, with random-choice baseline 1/9≈11%.

This checks alignment within the synthetic dataset, not general visual understanding or language generation. Text-to-image retrieval uses the same shared space but reverses the query/candidate roles.


In [ ]:
@torch.no_grad()
def retrieval_accuracy(n_queries=100):
    """Generate a random image and retrieve among nine class descriptions, checking the correct class."""
    rng = np.random.RandomState(123)
    correct = 0
    # Candidate text for all nine classes
    cand_ids = torch.tensor([make_text(c, s) for c in COLORS for s in SHAPES])
    cand_emb = F.normalize(txt_enc(cand_ids), dim=-1)     # [9, d]
    for _ in range(n_queries):
        c = rng.choice(COLORS)
        s = rng.choice(SHAPES)
        q_img = torch.tensor(make_image(c, s, rng=rng)).permute(2, 0, 1).unsqueeze(0)
        q_emb = F.normalize(img_enc(q_img), dim=-1)       # [1, d]
        hit = (q_emb @ cand_emb.t()).argmax().item()      # most similar description
        true = COLORS.index(c) * len(SHAPES) + SHAPES.index(s)
        correct += int(hit == true)
    return correct / n_queries


acc = retrieval_accuracy(100)
print(f"Image-text retrieval accuracy: {acc*100:.0f}%   (random baseline {100/9:.0f}%)")
print("\nKey observation: no explicit image-to-sentence pairing label was supplied;")
print("  Contrastive learning alone pulls both towers into one space, so a description can retrieve its image.")
print("  This is the origin of the VLM visual frontend: a CLIP-trained image encoder learns to understand images.")


### 11.6 CLIP and Generative VLMs

- **CLIP-style training** aligns pooled image/text representations.
- **Patch embedding / ViT** is one possible image-encoder structure.
- **A projector** maps visual features into the LLM's expected representation.
- **The language model** uses those features through visual tokens or another fusion architecture.

Contrastively pretrained features can be useful starting points, but not all VLMs use CLIP ViT or the same two-stage training recipe. Our toy uses a CNN, simple text pooling, fixed temperature, and nine classes. Original CLIP's reported scale includes 400 million pairs and batch size 32,768, with several image architectures. The shared contrastive idea is similar; the full implementations are not identical.


## Summary

- VLMs combine visual and language information; generative VLMs can answer in text.
- A common route encodes images into continuous visual vectors. Cross-attention, Q-Former, and other interfaces provide alternatives.
- In the 224×224/16×16 example, 14×14=196 patches. Real patch sizes, crops, resolutions, and resamplers vary.
- Patch embedding maps pixels to vectors; a projector makes vision features useful to the language stack.
- Concatenated visual tokens lengthen the language sequence. Cross-attention keeps the text sequence separate but adds its own work and feature storage.
- Classic LLaVA-style training aligns a connector, then tunes it with the LLM. Freezing policies depend on the recipe.

| Concept | Meaning |
|:---|:---|
| Patch | A local image region, e.g. 16×16 pixels |
| Patch embedding | A layer projecting patch pixels into vectors |
| Visual token | A visual representation passed to the multimodal model |
| Projector | Maps visual features to the language interface |
| Cross-attention | Text Q attends to visual K/V |
| Q-Former | Learned queries extract compact visual representations |

The central idea is to convert pixels into representations the language model can use. The full GPT-4V architecture is not public, so it is not evidence for one particular open architecture.

References: [ViT documentation](https://huggingface.co/docs/transformers/main/model_doc/vit), [LLaVA](https://arxiv.org/abs/2304.08485), [GPT-4V system card](https://cdn.openai.com/papers/GPTV_System_Card.pdf).


## Exercises

Ask AI for hints or checks if helpful, but work through the solutions.

### Exercise 1: Change the Patch Size

Change `patch_size` from 16 to 32 and inspect the visual token count.

Hint: `num_patches = (img_size // patch_size) ** 2`.

### Exercise 2: Implement a Linear Projector

Map 1,024-dimensional visual features into a 4,096-dimensional language representation with `nn.Linear`. The input is `[2, 196, 1024]`; the output's last dimension should be 4,096.

### Exercise 3: Two-Stage Trainable Parameters

Assume a 300M vision encoder, 7B LLM, and 8M projector. Stage 1 trains only the projector: 8M. Stage 2 trains projector + LLM: 8M+7,000M=7,008M. Calculate the trainable fraction of the complete model at each stage.


In [ ]:
patch_emb_32 = PatchEmbedding(img_size=224, patch_size=____, d_model=128)
assert patch_emb_32.num_patches == ____
print("Passed: you understand that larger patches produce fewer visual tokens.")


In [ ]:
simple_projector = nn.Linear(____, ____)
x = torch.randn(2, 196, 1024)
y = simple_projector(x)
assert y.shape == (2, 196, 4096)
print("Passed: you aligned one representation from vision space to language space.")


In [ ]:
vision_params = 300_000_000
llm_params = 7_000_000_000
projector_params = 8_000_000

stage1_trainable = ____
stage2_trainable = ____

assert stage1_trainable == 8_000_000
assert stage2_trainable == 7_008_000_000
print("Passed: you understand why VLM training freezes and trains components in stages.")


## References

- Radford et al., [Learning Transferable Visual Models (CLIP)](https://arxiv.org/abs/2103.00020), 2021.
- Dosovitskiy et al., [An Image is Worth 16×16 Words (ViT)](https://arxiv.org/abs/2010.11929), 2020.
- Liu et al., [Visual Instruction Tuning (LLaVA)](https://arxiv.org/abs/2304.08485), 2023.
